# Data Splitting for Classification and Risk Rating Tasks

This notebook performs a stratified 80/20 train/test split on `complaints_processed_full.csv`.

**Key Points:**
- Stratified by `product` category to maintain class distribution
- Used by all three tasks (Topic Modelling, Classification, Risk Rating)
- Train set: ~130K complaints (80%)
- Test set: ~32K complaints (20%)

**Output Files:**
- `data/train_indices.csv` - Row indices for train set
- `data/test_indices.csv` - Row indices for test set
- `data/train_data.csv` - Full train set
- `data/test_data.csv` - Full test set
- `data/split_info.txt` - Split metadata and class distribution

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

## Load Data

In [ ]:
# Load the processed complaints dataset
df = pd.read_csv('../data/complaints_processed_full.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

## Check Class Distribution (Before Split)

In [ ]:
# Display product category distribution
product_dist = df['product'].value_counts()
product_dist_pct = df['product'].value_counts(normalize=True) * 100

print("Product Category Distribution (Before Split):")
print("="*60)
dist_df = pd.DataFrame({
    'Product': product_dist.index,
    'Count': product_dist.values,
    'Percentage': product_dist_pct.values
})
print(dist_df.to_string(index=False))
print(f"\nTotal: {len(df)} complaints")

## Perform Stratified Train/Test Split

In [ ]:
# Perform stratified 80/20 split
# Random state = 42 for reproducibility
train_indices, test_indices = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    stratify=df['product'],
    random_state=42
)

# Create train and test datasets
train_df = df.iloc[train_indices].reset_index(drop=True)
test_df = df.iloc[test_indices].reset_index(drop=True)

print(f"Train set size: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test set size: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

## Validate Class Distribution (After Split)

In [ ]:
# Compare class distributions before and after split
print("\nClass Distribution Comparison:")
print("="*80)

train_dist = train_df['product'].value_counts(normalize=True) * 100
test_dist = test_df['product'].value_counts(normalize=True) * 100
original_dist = df['product'].value_counts(normalize=True) * 100

comparison_df = pd.DataFrame({
    'Product': original_dist.index,
    'Original %': original_dist.values,
    'Train %': train_dist.values,
    'Test %': test_dist.values
})

print(comparison_df.to_string(index=False))
print(f"\n✓ Stratification successful! Class distribution maintained across train/test splits.")

## Save Split Data and Indices

In [ ]:
output_dir = '../data'

# Save train and test indices
pd.DataFrame({'train_index': train_indices}).to_csv(
    f'{output_dir}/train_indices.csv', index=False
)
pd.DataFrame({'test_index': test_indices}).to_csv(
    f'{output_dir}/test_indices.csv', index=False
)
print(f"✓ Saved train_indices.csv ({len(train_indices)} rows)")
print(f"✓ Saved test_indices.csv ({len(test_indices)} rows)")

# Save train and test datasets
train_df.to_csv(f'{output_dir}/train_data.csv', index=False)
test_df.to_csv(f'{output_dir}/test_data.csv', index=False)
print(f"✓ Saved train_data.csv ({len(train_df)} rows)")
print(f"✓ Saved test_data.csv ({len(test_df)} rows)")

## Save Split Metadata

In [ ]:
# Create split metadata file
metadata = f"""DATA SPLIT METADATA
{'='*80}

Source Dataset: complaints_processed_full.csv
Total Records: {len(df)}

SPLIT PARAMETERS:
- Split Ratio: 80% train / 20% test
- Stratification: By 'product' category
- Random State: 42 (for reproducibility)

SPLIT RESULTS:
- Train set size: {len(train_df)} ({len(train_df)/len(df)*100:.2f}%)
- Test set size: {len(test_df)} ({len(test_df)/len(df)*100:.2f}%)

CLASS DISTRIBUTION BY PRODUCT:
\n{comparison_df.to_string(index=False)}

OUTPUT FILES:
1. train_indices.csv - CSV of row indices for train set
2. test_indices.csv - CSV of row indices for test set
3. train_data.csv - Full train dataset with all columns
4. test_data.csv - Full test dataset with all columns
5. split_info.txt - This metadata file

USAGE INSTRUCTIONS:

For members using indices:
  train_indices = pd.read_csv('data/train_indices.csv')['train_index'].values
  test_indices = pd.read_csv('data/test_indices.csv')['test_index'].values
  train_df = df.iloc[train_indices]
  test_df = df.iloc[test_indices]

For members using pre-split data:
  train_df = pd.read_csv('data/train_data.csv')
  test_df = pd.read_csv('data/test_data.csv')

NOTE: All team members MUST use this same split for consistency and comparability.
{'='*80}
"""

with open(f'{output_dir}/split_info.txt', 'w') as f:
    f.write(metadata)

print("✓ Saved split_info.txt")
print("\n" + metadata)

## Summary

In [ ]:
print("\n" + "="*80)
print("DATA SPLITTING COMPLETE")
print("="*80)
print(f"""
✓ Stratified 80/20 split created
✓ Train set: {len(train_df):,} complaints (80%)
✓ Test set: {len(test_df):,} complaints (20%)
✓ Class distribution maintained across all categories

📁 Output files saved to data/ directory:
  - train_indices.csv
  - test_indices.csv
  - train_data.csv
  - test_data.csv
  - split_info.txt

✅ All team members should now use this split for Tasks 1-3
""")